# CMPT 413 / 713 Tutorial 6
## Text classification with PyTorch using word embeddings (cont'd)

In [4]:
# Load the AG NEWS dataset from the torchtext library
import torch
import torchtext

train_dataset, val_dataset = torchtext.datasets.AG_NEWS()

## Basic preprocessing of text
1. Lower Case: 'He is walking around the park.' -> 'he is walking around the park.'
2. Remove Punctuation: 'he is walking around the park.' -> 'he is walking around the park'
3. Stem: 'he is walking around the park' -> 'he is walk around the park'

In [5]:
# Some basic preprocessing of text dataset
import io
import nltk
import string
import numpy as np
from tqdm import tqdm
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize
  
stemmer = PorterStemmer()

nltk.download('punkt')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('stopwords')

# Function to remove punctuation
def remove_punctuation(text):
    punctuationfree="".join([i for i in text if i not in string.punctuation])
    return punctuationfree

# Function to tokenize text
def tokenization(text):
    tokens = word_tokenize(text)
    return tokens

# Stem function for the words
def stem(text):
    stem_text = [stemmer.stem(word) for word in text]
    return stem_text

train_text = [data[1] for data in train_dataset]
val_text = [data[1] for data in val_dataset]

train_len = []
val_len = []

train_labels = np.array([item[0] for item in train_dataset]) - 1
val_labels = np.array([item[0] for item in val_dataset]) - 1

print('Original Text:')
print(train_text[0])
temp = train_text[0].lower()
print()
print('Lower Case:')
print(temp)
temp = remove_punctuation(temp)
print()
print('Remove Punctuation:')
print(temp)
temp = tokenization(temp)
print()
print('Tokenization:')
print(temp)
temp = stem(temp)
print()
print('Stem:')
print(temp)

for i in tqdm(range(len(train_text))):
    train_text[i] = train_text[i].lower()
    train_text[i] = remove_punctuation(train_text[i])
    train_text[i] = tokenization(train_text[i])
    train_text[i] = stem(train_text[i])
    train_len.append(len(train_text[i]))
    
for i in tqdm(range(len(val_text))):
    val_text[i] = val_text[i].lower()
    val_text[i] = remove_punctuation(val_text[i])
    val_text[i] = tokenization(val_text[i])
    val_text[i] = stem(val_text[i])
    val_len.append(len(val_text[i]))
    
train_len = np.array(train_len)
val_len = np.array(val_len)

[nltk_data] Downloading package punkt to /home/eamonn/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to /home/eamonn/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /home/eamonn/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package stopwords to /home/eamonn/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Original Text:
Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\band of ultra-cynics, are seeing green again.

Lower Case:
wall st. bears claw back into the black (reuters) reuters - short-sellers, wall street's dwindling\band of ultra-cynics, are seeing green again.

Remove Punctuation:
wall st bears claw back into the black reuters reuters  shortsellers wall streets dwindlingband of ultracynics are seeing green again

Tokenization:
['wall', 'st', 'bears', 'claw', 'back', 'into', 'the', 'black', 'reuters', 'reuters', 'shortsellers', 'wall', 'streets', 'dwindlingband', 'of', 'ultracynics', 'are', 'seeing', 'green', 'again']

Stem:
['wall', 'st', 'bear', 'claw', 'back', 'into', 'the', 'black', 'reuter', 'reuter', 'shortsel', 'wall', 'street', 'dwindlingband', 'of', 'ultracyn', 'are', 'see', 'green', 'again']


100%|█████████████████████████████████████| 7600/7600 [00:04<00:00, 1875.28it/s]


## Convert words to integers
1. We want to build a vocabulary dictionary that converts words to integers eg. 'I' -> 13.
2. We want to avoid words that don't appear often. Here we use an arbitrary threshold of at least 10 occurences.
3. Add \<PAD\> to pad sentences to have the same length (for Batching).
4. Add \<UNK\> for words that are not in vocab (occurence < 10).

In [6]:
# Get word to number mapping for word counts that are over 10
from collections import Counter

counter = Counter()
for sentence in tqdm(train_text):
    counter.update(sentence)

# Here we only get words that appears more than 10 times
counter = Counter({k: c for k, c in counter.items() if c >= 10})
vocab = list(counter.keys())

vocab_to_id = {}
vocab_to_id['<PAD>'] = 0
vocab_to_id['<UNK>'] = 1

counter = 2
for word in vocab:
    vocab_to_id[word] = counter
    counter += 1
vocab_size = counter
print('Vocab Size:', counter)

100%|███████████████████████████████| 120000/120000 [00:00<00:00, 222883.14it/s]

Vocab Size: 15089


## Convert sentences to chunks
1. Split sentences into chunks, each has 4 words
2. Prepare ground truth labels.

Example: This is a simple tutorial.

data:
1. \<PAD\>,\<PAD\>,\<PAD\>,This
2. \<PAD\>,\<PAD\>,This,is
3. \<PAD\>,This,is,a
4. This,is,a,simple
    
GT labels:
1. is
2. a
3. simple
4. tutorial


In [15]:
chunked_train_data = []
chunked_val_data = []


def convert_sentences_to_chunks(sentences, chunk_size):
    chunked_data = []
    chunked_labels = []
    for data in tqdm(sentences):
        # add paddings for the first (chunk_size - 1) chunks
        for i in range(chunk_size - 1):
            gt_label = data[i+1]
            if gt_label not in vocab_to_id:
                # word is not in vocabulary, skip
                continue
            chunked_labels.append(gt_label)
            tmp = ['<PAD>'] * (chunk_size - 1 - i) + data[0:i+1]
            chunked_data.append(tmp)
            
        for i in range(len(data) - chunk_size):
            gt_label = data[i+chunk_size]
            if gt_label not in vocab_to_id:
                # word is not in vocabulary, skip
                continue
            chunked_labels.append(data[i+chunk_size])
            chunked_data.append(data[i:i+chunk_size])
            
    assert len(chunked_data) == len(chunked_labels)
    return chunked_data, chunked_labels

chunk_size = 4

chunked_train_data, chunked_train_labels = convert_sentences_to_chunks(train_text, chunk_size)
chunked_val_data, chunked_val_labels = convert_sentences_to_chunks(val_text, chunk_size)

print("Original sentence (tokens):")
print(train_text[0])
print("\nChunked sentence:")
print(chunked_train_data[0:len(train_text[0]) - chunk_size])
print("\nGT labels:")
print(chunked_train_labels[0:len(train_text[0]) - chunk_size])

100%|████████████████████████████████████| 7600/7600 [00:00<00:00, 71346.07it/s]

Original sentence (tokens):
['wall', 'st', 'bear', 'claw', 'back', 'into', 'the', 'black', 'reuter', 'reuter', 'shortsel', 'wall', 'street', 'dwindlingband', 'of', 'ultracyn', 'are', 'see', 'green', 'again']

Chunked sentence:
[['<PAD>', '<PAD>', '<PAD>', 'wall'], ['<PAD>', '<PAD>', 'wall', 'st'], ['<PAD>', 'wall', 'st', 'bear'], ['wall', 'st', 'bear', 'claw'], ['st', 'bear', 'claw', 'back'], ['bear', 'claw', 'back', 'into'], ['claw', 'back', 'into', 'the'], ['back', 'into', 'the', 'black'], ['into', 'the', 'black', 'reuter'], ['black', 'reuter', 'reuter', 'shortsel'], ['reuter', 'reuter', 'shortsel', 'wall'], ['shortsel', 'wall', 'street', 'dwindlingband'], ['street', 'dwindlingband', 'of', 'ultracyn'], ['dwindlingband', 'of', 'ultracyn', 'are'], ['of', 'ultracyn', 'are', 'see'], ['ultracyn', 'are', 'see', 'green']]

GT labels:
['st', 'bear', 'claw', 'back', 'into', 'the', 'black', 'reuter', 'reuter', 'wall', 'street', 'of', 'are', 'see', 'green', 'again']


## Custom dataset loader
1. Converts words to the corresponding vocabulary index.
2. Pad or cut sentences to have the same length of 50. (for Batching)

In [16]:
# Custom dataset loader
from torch.utils.data import Dataset, DataLoader

class TextDataset(Dataset):
    def __init__(self, text_dataset, labels, vocab_to_id):
        super(TextDataset, self).__init__()
        self.text_dataset = text_dataset
        self.labels = labels
        self.vocab_to_id = vocab_to_id

    def __len__(self):
        return len(self.text_dataset)

    def __getitem__(self, idx):
        text = self.text_dataset[idx]
        label = self.labels[idx]
        if label in self.vocab_to_id:
            label = self.vocab_to_id[label]
        else:
            label = self.vocab_to_id['<UNK>']
        tokens = []
        
        for counter in range(len(text)):
            if text[counter] in self.vocab_to_id.keys():
                tokens.append(self.vocab_to_id[text[counter]])
            else:
                tokens.append(self.vocab_to_id['<UNK>'])
        return np.array(tokens), np.array(label)
    
train_dataset = TextDataset(chunked_train_data, chunked_train_labels, vocab_to_id)
train_dataloader = DataLoader(train_dataset, batch_size=10240, shuffle=True, num_workers=4)

val_dataset = TextDataset(chunked_val_data, chunked_val_labels, vocab_to_id)
val_dataloader = DataLoader(val_dataset, batch_size=10240, num_workers=4)

print(chunked_train_data[0])
print(train_dataset[0][0])
print(train_dataset[0][1])

['<PAD>', '<PAD>', '<PAD>', 'wall']
[0 0 0 2]
3


## Define neural network
### We add an embedding layer compared to the last tutorial.
### The embeddings layer takes integers and turns them into vector embeddings.
### $$i\in\mathbb{Z}$$
### $$\text{embedding}(i)\in\mathbb{R}^{256}$$

In [17]:
from torch import nn
from torch.optim import Adam
from torch.nn import functional as F
from sklearn.metrics import accuracy_score

class TextClassifier(nn.Module):
    def __init__(self, chunk_size):
        super(TextClassifier, self).__init__()
        self.embedding_layer = nn.Embedding(vocab_size, 256, padding_idx=0)
        self.seq = nn.Sequential(
            nn.Linear(256 * chunk_size, 512),
            nn.ReLU(),
            nn.Linear(512, 1024),
            nn.ReLU(),
            nn.Linear(1024, vocab_size)
        )

    def forward(self, x):
        embeddings = self.embedding_layer(x)
        
        # we concatenate word embeddings to get a vector        
        embeddings = embeddings.flatten(start_dim=1, end_dim=2)
        
        logits = self.seq(embeddings)
        return logits
    
classifier = TextClassifier(chunk_size).cuda()
print('Sentence tokens:', val_dataset[0][0])
print()
print('Sentence tokens shape:', val_dataset[0][0].shape)
print()
embeddings = classifier.embedding_layer(torch.from_numpy(val_dataset[0][0]).cuda())
print('Shape after embedding layer:', embeddings.shape)

Sentence tokens: [   0    0    0 2072]

Sentence tokens shape: (4,)

Shape after embedding layer: torch.Size([4, 256])


## Training function (same as tutorial 4)

In [18]:
loss_fn = nn.CrossEntropyLoss()
optimizer = Adam(classifier.parameters(), lr=0.001)


def cal_loss_and_accuracy(model, loss_fn, val_data_loader):
    with torch.no_grad():
        y_actual, y_preds, losses, perplexities = [],[],[],[]
        for x, y in val_data_loader:
            y_cuda = y.cuda()
            x_cuda = x.cuda()
            preds = model(x_cuda)
            loss = loss_fn(preds, y_cuda)
            losses.append(loss.item())
            y_actual.append(y_cuda)
            y_preds.append(preds.argmax(dim=-1))
            perplexities.append(np.power(2, loss.item()))

        y_actual = torch.cat(y_actual)
        y_preds = torch.cat(y_preds)
        perplexities = np.array(perplexities)

        print("Val Loss: {:.3f}".format(torch.tensor(losses).mean()))
        print("Val Perplexity: {:.3f}".format(perplexities.mean()))
        print("Val Accuracy: {:.3f}".format(accuracy_score(y_actual.cpu().numpy(), y_preds.cpu().numpy())))



# train the model 5 epochs
for i in range(5):
    losses = []
    for x, y in tqdm(train_dataloader):
        
        # forward
        y_preds = classifier(x.cuda())
        
        # calculate losses
        loss = loss_fn(y_preds, y.cuda())
        losses.append(loss.item())
        
        # empty old gradients
        optimizer.zero_grad()
        
        # calculate gradients and backward
        loss.backward()
        
        # update all parameters
        optimizer.step()

    print("Train Loss : {:.3f}".format(torch.tensor(losses).mean()))
    cal_loss_and_accuracy(classifier, loss_fn, val_dataloader)

100%|█████████████████████████████████████████| 415/415 [01:02<00:00,  6.60it/s]

Train Loss : 6.258


Val Loss: 5.641
Val Perplexity: 49.934
Val Accuracy: 0.169


100%|█████████████████████████████████████████| 415/415 [01:01<00:00,  6.73it/s]

Train Loss : 5.321


...

Train Loss : 4.628


Val Loss: 5.155
Val Perplexity: 35.685
Val Accuracy: 0.215


100%|█████████████████████████████████████████| 415/415 [01:03<00:00,  6.52it/s]

Train Loss : 4.399


Val Loss: 5.166
Val Perplexity: 35.966
Val Accuracy: 0.220


## Valiation accuracy function (same as tutorial 4)

In [ ]:
from sklearn.metrics import classification_report

y_actual, y_preds = [], []

# test the model
with torch.no_grad():
    for x, y in val_dataloader:
        preds = classifier(x.cuda())
        
        y_preds.append(F.softmax(preds, dim=-1).argmax(dim=-1).cpu())
        y_actual.append(y)
    
    y_preds, y_actual = torch.cat(y_preds), torch.cat(y_actual)
    
    
# print results
print("Accuracy : {}".format(accuracy_score(y_actual, y_preds)))
print("\nClassification Report : ")
print(classification_report(y_actual, y_preds))

## Why is performance bad?
1. Not enough data. Only 120000 in training dataset with a vocab size of 15089. Neural Networks needs more data points compared to traditional methods.
2. We simply concatenate the word embeddings for partial sentences.

## How do we improve?
1. Use pretrained word embeddings such as word2vec instead of learning it ourselves. Better embeddings will help the learning with less samples. (Tutorial 5)
2. Specialized architectures such as RNN to process sentences instead. (Next tutorial)